In [1]:
import os
import sys

# Setup path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)
print(f"✅ Working directory: {os.getcwd()}")

✅ Working directory: c:\Users\lucas\workspace\github\fase-5


# 🤖 Análise Exploratória e Modelagem - Passos Mágicos

## 🎯 Objetivo

Pipeline completo: carregamento → limpeza → features → modelagem

**Etapas**:
1. Carregamento dos dados
2. Limpeza e tratamento de missing values
3. Criação de target binário (risco de defasagem)
4. Feature engineering com funções reutilizáveis
5. Seleção de features e preparação para modelagem
6. Treinamento de Random Forest com validação cruzada
7. Avaliação e análise de feature importance

## 🧠 Lógica
- Target: Defasagem < 0 = Risco (1), caso contrário = Sem Risco (0)
- StratifiedKFold mantém proporção de classes
- Class weight balanceado para desbalanceamento

In [2]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from scripts.visualization import plot_feature_importance
from src.data_cleaning import load_data
from src.feature_engineering import create_features

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✅ Imports concluídos")

✅ Imports concluídos


## 1. Carregamento e Limpeza

In [ ]:
DATA_PATH = "../app/data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

try:
    df = load_data(DATA_PATH)
    print(f"✅ Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
except Exception as e:
    print(f"⚠️ Erro ao carregar Excel, usando CSV fallback: {e}")
    df = pd.read_csv("../app/data/processed/df_model_2022.csv")
    print(f"✅ CSV carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")

# Limpeza
df = clean_dataframe(df)
print(f"✅ Após limpeza: {df.shape[0]} linhas")

## 2. Criação do Target

In [ ]:
# Identificar coluna de defasagem
defasagem_col = "Defasagem" if "Defasagem" in df.columns else "DEFASAGEM"

if defasagem_col in df.columns:
    df["target_risco"] = (df[defasagem_col] < 0).astype(int)
    print("✅ Target criado:")
    print(f"   Sem Risco: {(df['target_risco'] == 0).sum()}")
    print(f"   Com Risco: {(df['target_risco'] == 1).sum()}")
else:
    print("⚠️ Coluna de defasagem não encontrada")

## 3. Feature Engineering

In [ ]:
try:
    df = create_features(df)
    print("✅ Features engenheirizadas")
except Exception as e:
    print(f"⚠️ Erro no feature engineering: {e}")
    print("   Continuando com features originais")

## 4. Tratamento de Missing Values

In [ ]:
# Remover colunas com > 70% missing
missing_pct = df.isna().sum() / len(df) * 100
cols_to_drop = missing_pct[missing_pct > 70].index

df = df.drop(columns=cols_to_drop)
print(f"❌ Removidas {len(cols_to_drop)} colunas")

# Preencher com média/moda conforme tipo
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    if col not in ["NOME", "target_risco"]:
        mode_val = df[col].mode()
        df[col] = df[col].fillna(mode_val[0] if len(mode_val) > 0 else "Unknown")

print("✅ Missing values tratados")

## 5. Preparação para Modelagem

In [ ]:
# Separar X e y
X = df.select_dtypes(include=[np.number]).drop(columns=["target_risco"], errors="ignore")
y = df["target_risco"] if "target_risco" in df.columns else None

if y is not None:
    print("✅ Dataset preparado:")
    print(f"   X: {X.shape[0]} × {X.shape[1]}")
    print(f"   y: {len(y)} labels")
    print(f"   Risco: {(y.sum() / len(y) * 100):.1f}%")
else:
    print("⚠️ Target não disponível")

## 6. Train/Test Split

In [ ]:
if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 7. Treinamento

In [ ]:
if y is not None:
    modelo = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    modelo.fit(X_train_scaled, y_train)

    # Predições
    y_pred = modelo.predict(X_test_scaled)
    y_pred_proba = modelo.predict_proba(X_test_scaled)[:, 1]

    print("✅ Modelo treinado")
    print("\n📊 Relatório:")
    print(classification_report(y_test, y_pred, target_names=["Sem Risco", "Risco"]))
    print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

## 8. Feature Importance

In [ ]:
if y is not None:
    feature_importance = pd.DataFrame(
        {"feature": X.columns, "importance": modelo.feature_importances_}
    ).sort_values("importance", ascending=False)

    print("🏆 Top 10 Features:")
    print(feature_importance.head(10).to_string(index=False))

    # Plot
    plot_feature_importance(feature_importance.head(10), title="Top 10 Features")

print("\n✅ Pipeline concluído")